# 05B: EXTRACT NETWORK FEATURES

This notebook extracts raster-based features at network nodes for LST prediction. These nodes cover the full study area including locations without GSV imagery.

## MODULE SETUP

In [ ]:
# Mount Google Drive.
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
import os

# Set working directory to project folder.
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/hot_hem"
os.chdir(BASE_DIR)

print(f"Working directory: {BASE_DIR}")

Working directory: /content/drive/MyDrive/Colab Notebooks/hot_hem


## IMPORT SETUP

In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import rasterio
import rasterio.warp
from tqdm import tqdm
from skimage.feature import graycomatrix, graycoprops

## PATH CONFIGURATION

In [ ]:
# Input paths.
NETWORK_NODES = Path("data/processing/network/network_nodes.csv")

# Raster inputs.
RASTER_DIR = Path("data/inputs/raster")
LANDSAT = RASTER_DIR / "LANDSAT_composite_raster.tif"
PALSAR = RASTER_DIR / "JAXA_PALSAR-2_2024_composite_bands.tif"
DSM = RASTER_DIR / "JAXA_DSM_ALPSMLC30_N010_composite_bands.tif"
LANDCOVER = RASTER_DIR / "JAXA_LULC_N10E106_2020_v23.09_10m.tif"

# Output paths.
OUT_DIR = Path("data/outputs/features")
OUT_FILE = OUT_DIR / "network_nodes_with_raster_features.csv"

# Create output directory.
OUT_DIR.mkdir(parents = True, exist_ok = True)

print("PATH CONFIGURATION")
print(f"Network nodes: {NETWORK_NODES}")
print(f"Raster directory: {RASTER_DIR}")
print(f"Output file: {OUT_FILE}")

PATH CONFIGURATION
Network nodes: data/processing/network/network_nodes.csv
Raster directory: data/inputs/raster
Output file: data/outputs/features/network_nodes_with_raster_features.csv


## LOAD NETWORK NODES

In [ ]:
# Load network nodes.
df_nodes = pd.read_csv(NETWORK_NODES)

print(f"Loaded {len(df_nodes)} network nodes.")
print(f"Columns: {list(df_nodes.columns)}")
df_nodes.head()

Loaded 28437 network nodes.
Columns: ['osmid', 'y', 'x', 'street_count', 'highway', 'ref', 'railway', 'geometry']


,osmid,y,x,street_count,highway,ref,railway,geometry
0,366367367,10.813901,106.732449,1,NaN,NaN,NaN,POINT (106.7324492 10.8139013)
1,366368182,10.742144,106.670913,3,NaN,NaN,NaN,POINT (106.6709134 10.7421444)
2,366368194,10.790241,106.709544,3,NaN,NaN,NaN,POINT (106.7095438 10.7902412)
3,366368313,10.754628,106.684140,4,NaN,NaN,NaN,POINT (106.6841396 10.7546276)
4,366368540,10.742907,106.672759,4,NaN,NaN,NaN,POINT (106.6727592 10.742907)


In [ ]:
# Ensure lat/lon columns exist.
# OSMnx exports as x/y columns.
if "lon" not in df_nodes.columns and "x" in df_nodes.columns:
    df_nodes["lon"] = df_nodes["x"]
if "lat" not in df_nodes.columns and "y" in df_nodes.columns:
    df_nodes["lat"] = df_nodes["y"]

print(f"Coordinate range: lat [{df_nodes['lat'].min():.4f}, {df_nodes['lat'].max():.4f}]")
print(f"Coordinate range: lon [{df_nodes['lon'].min():.4f}, {df_nodes['lon'].max():.4f}]")

Coordinate range: lat [10.6857, 10.8186]
Coordinate range: lon [106.5902, 106.8108]


## HELPER FUNCTIONS

In [ ]:
def calculate_sky_view_factor(dem_window, cell_size = 30):
    """
    Calculate approximate Sky View Factor from DEM window.
    SVF measures openness to sky based on surrounding terrain.
    Returns value between 0 and 1.
    """
    center_idx = dem_window.shape[0] // 2
    center_elev = dem_window[center_idx, center_idx]

    if np.isnan(center_elev):
        return np.nan

    # Calculate horizon angles in 8 directions.
    angles = []
    directions = [
        (-1, 0), (-1, 1), (0, 1), (1, 1),
        (1, 0), (1, -1), (0, -1), (-1, -1)
    ]

    for dr, dc in directions:
        max_angle = 0

        for step in range(1, center_idx + 1):
            r = center_idx + dr * step
            c = center_idx + dc * step

            if 0 <= r < dem_window.shape[0] and 0 <= c < dem_window.shape[1]:
                elev = dem_window[r, c]

                if not np.isnan(elev):
                    dist = step * cell_size
                    angle = np.arctan((elev - center_elev) / dist)
                    max_angle = max(max_angle, angle)

        angles.append(max_angle)

    # SVF approximation.
    mean_angle = np.mean(angles)
    svf = np.cos(mean_angle) ** 2

    return float(svf)

## EXTRACT LANDSAT FEATURES

In [ ]:
start_time = time.time()

# Initialize feature columns.
df_nodes["lst_kelvin"] = np.nan
df_nodes["lst_celsius"] = np.nan
df_nodes["ndvi"] = np.nan
df_nodes["emissivity"] = np.nan

# Extract coordinate arrays.
lons = df_nodes["lon"].values
lats = df_nodes["lat"].values

# Open Landsat composite.
with rasterio.open(LANDSAT) as src:
    print("LANDSAT RASTER PROPERTIES")
    print(f"Bands: {src.count}")
    print(f"Shape: {src.shape}")
    print(f"CRS: {src.crs}")

    # Transform coordinates.
    xs, ys = rasterio.warp.transform("EPSG:4326", src.crs, lons, lats)

    # Convert to pixel indices.
    rows = []
    cols = []

    for x, y in zip(xs, ys):
        try:
            r, c = src.index(x, y)
            rows.append(r)
            cols.append(c)
        except:
            rows.append(-1)
            cols.append(-1)

    rows = np.array(rows)
    cols = np.array(cols)

    print("Reading bands.")

    # Read bands.
    lst_kelvin_array = src.read(1).astype(float)
    emissivity_array = src.read(2).astype(float)
    ndvi_array = src.read(3).astype(float)
    qa_array = src.read(4).astype(int)

    # Handle nodata.
    if src.nodata is not None:
        lst_kelvin_array[lst_kelvin_array == src.nodata] = np.nan
        emissivity_array[emissivity_array == src.nodata] = np.nan
        ndvi_array[ndvi_array == src.nodata] = np.nan

    lst_kelvin_array[lst_kelvin_array == 0] = np.nan
    emissivity_array[emissivity_array == 0] = np.nan

    print("Extracting values.")

    # Create validity mask.
    valid_mask = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)

    # Extract values.
    lst_kelvin = np.full(len(df_nodes), np.nan)
    emissivity = np.full(len(df_nodes), np.nan)
    ndvi = np.full(len(df_nodes), np.nan)
    qa_vals = np.full(len(df_nodes), 0)

    lst_kelvin[valid_mask] = lst_kelvin_array[rows[valid_mask], cols[valid_mask]]
    emissivity[valid_mask] = emissivity_array[rows[valid_mask], cols[valid_mask]]
    ndvi[valid_mask] = ndvi_array[rows[valid_mask], cols[valid_mask]]
    qa_vals[valid_mask] = qa_array[rows[valid_mask], cols[valid_mask]]

    # Convert LST to Celsius.
    lst_celsius = lst_kelvin - 273.15

    # Apply cloud mask.
    cloud_mask = (qa_vals & (1 << 1)) | (qa_vals & (1 << 3)) | (qa_vals & (1 << 4))
    lst_celsius[cloud_mask != 0] = np.nan

    # Store in dataframe.
    df_nodes["lst_kelvin"] = lst_kelvin
    df_nodes["lst_celsius"] = lst_celsius
    df_nodes["ndvi"] = ndvi
    df_nodes["emissivity"] = emissivity

elapsed = time.time() - start_time
print(f"Landsat extraction complete in {elapsed:.1f} seconds.")
print(f"LST: {df_nodes['lst_celsius'].notna().sum()} / {len(df_nodes)} nodes")

LANDSAT RASTER PROPERTIES
Bands: 4
Shape: (654, 947)
CRS: EPSG:32648
Reading bands.
Extracting values.
Landsat extraction complete in 4.7 seconds.
LST: 28332 / 28437 nodes


## EXTRACT PALSAR FEATURES

In [ ]:
start_time = time.time()

# Initialize feature columns.
df_nodes["palsar_hh_db"] = np.nan
df_nodes["palsar_hv_db"] = np.nan
df_nodes["palsar_hv_hh_ratio"] = np.nan
df_nodes["palsar_glcm_contrast"] = np.nan
df_nodes["palsar_glcm_homogeneity"] = np.nan
df_nodes["palsar_glcm_energy"] = np.nan

# Open PALSAR composite.
with rasterio.open(PALSAR) as src:
    print("PALSAR RASTER PROPERTIES")
    print(f"Bands: {src.count}")
    print(f"Shape: {src.shape}")

    # Transform coordinates.
    xs, ys = rasterio.warp.transform("EPSG:4326", src.crs, lons, lats)

    # Convert to pixel indices.
    rows = []
    cols = []

    for x, y in zip(xs, ys):
        try:
            r, c = src.index(x, y)
            rows.append(r)
            cols.append(c)
        except:
            rows.append(-1)
            cols.append(-1)

    rows = np.array(rows)
    cols = np.array(cols)
    valid_mask = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)

    print("Reading bands.")

    # Read HH and HV bands.
    hh_array = src.read(1).astype(float)
    hv_array = src.read(2).astype(float)

    # Handle nodata.
    if src.nodata is not None:
        hh_array[hh_array == src.nodata] = np.nan
        hv_array[hv_array == src.nodata] = np.nan

    print("Extracting backscatter values.")

    # Extract values.
    hh_dn = np.full(len(df_nodes), np.nan)
    hv_dn = np.full(len(df_nodes), np.nan)

    hh_dn[valid_mask] = hh_array[rows[valid_mask], cols[valid_mask]]
    hv_dn[valid_mask] = hv_array[rows[valid_mask], cols[valid_mask]]

    # Convert DN to dB.
    hh_db = 10 * np.log10(hh_dn ** 2) - 83.0
    hv_db = 10 * np.log10(hv_dn ** 2) - 83.0

    # Store values.
    df_nodes["palsar_hh_db"] = hh_db
    df_nodes["palsar_hv_db"] = hv_db
    df_nodes["palsar_hv_hh_ratio"] = hv_db / hh_db

    print(f"Backscatter extracted for {np.sum(~np.isnan(hh_db))} nodes.")

    # Calculate texture features.
    print("Computing texture features.")

    window_size = 5
    half_win = window_size // 2

    for i in tqdm(range(len(df_nodes)), desc = "Processing nodes"):
        if not valid_mask[i]:
            continue

        r = rows[i]
        c = cols[i]

        # Define window bounds.
        r_start = max(0, r - half_win)
        r_end = min(hh_array.shape[0], r + half_win + 1)
        c_start = max(0, c - half_win)
        c_end = min(hh_array.shape[1], c + half_win + 1)

        # Extract window.
        hh_window = hh_array[r_start:r_end, c_start:c_end].copy()
        hh_window_db = 10 * np.log10(hh_window ** 2 + 1e-10) - 83.0

        # Skip if too many NaN values.
        if np.sum(np.isnan(hh_window_db)) > hh_window_db.size * 0.5:
            continue

        # Normalize to uint8 for GLCM.
        hh_window_db = np.nan_to_num(hh_window_db, nan = np.nanmean(hh_window_db))
        hh_norm = ((hh_window_db - hh_window_db.min()) / (hh_window_db.max() - hh_window_db.min() + 1e-10) * 255).astype(np.uint8)

        # Compute GLCM.
        try:
            glcm = graycomatrix(hh_norm, [1], [0], levels = 256, symmetric = True, normed = True)

            df_nodes.at[i, "palsar_glcm_contrast"] = graycoprops(glcm, "contrast")[0, 0]
            df_nodes.at[i, "palsar_glcm_homogeneity"] = graycoprops(glcm, "homogeneity")[0, 0]
            df_nodes.at[i, "palsar_glcm_energy"] = graycoprops(glcm, "energy")[0, 0]
        except:
            continue

elapsed = time.time() - start_time
print(f"PALSAR extraction complete in {elapsed:.1f} seconds.")

PALSAR RASTER PROPERTIES
Bands: 5
Shape: (804, 1178)
Reading bands.
Extracting backscatter values.
Backscatter extracted for 28437 nodes.
Computing texture features.


Processing nodes: 100%|██████████| 28437/28437 [01:10<00:00, 400.91it/s]

PALSAR extraction complete in 74.2 seconds.


## EXTRACT DSM FEATURES

In [ ]:
start_time = time.time()

# Initialize feature columns.
df_nodes["elevation_m"] = np.nan
df_nodes["sky_view_factor"] = np.nan

# Open DSM composite.
with rasterio.open(DSM) as src:
    print("DSM RASTER PROPERTIES")
    print(f"Bands: {src.count}")
    print(f"Shape: {src.shape}")

    # Transform coordinates.
    xs, ys = rasterio.warp.transform("EPSG:4326", src.crs, lons, lats)

    # Convert to pixel indices.
    rows = []
    cols = []

    for x, y in zip(xs, ys):
        try:
            r, c = src.index(x, y)
            rows.append(r)
            cols.append(c)
        except:
            rows.append(-1)
            cols.append(-1)

    rows = np.array(rows)
    cols = np.array(cols)
    valid_mask = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)

    print("Reading elevation band.")

    # Read elevation.
    dem_array = src.read(1).astype(float)

    if src.nodata is not None:
        dem_array[dem_array == src.nodata] = np.nan

    print("Extracting elevation.")

    # Extract elevation.
    elevations = np.full(len(df_nodes), np.nan)
    elevations[valid_mask] = dem_array[rows[valid_mask], cols[valid_mask]]
    df_nodes["elevation_m"] = elevations

    print(f"Elevation extracted for {np.sum(~np.isnan(elevations))} nodes.")

    # Calculate Sky View Factor.
    print("Computing Sky View Factor.")

    window_size = 11
    half_win = window_size // 2

    for i in tqdm(range(len(df_nodes)), desc = "Processing nodes"):
        if not valid_mask[i]:
            continue

        r = rows[i]
        c = cols[i]

        # Define window bounds.
        r_start = max(0, r - half_win)
        r_end = min(dem_array.shape[0], r + half_win + 1)
        c_start = max(0, c - half_win)
        c_end = min(dem_array.shape[1], c + half_win + 1)

        # Extract window.
        dem_window = dem_array[r_start:r_end, c_start:c_end]

        # Calculate SVF if enough valid pixels.
        if np.sum(~np.isnan(dem_window)) >= 9:
            svf = calculate_sky_view_factor(dem_window, cell_size = 30)
            df_nodes.at[i, "sky_view_factor"] = svf

elapsed = time.time() - start_time
print(f"DSM extraction complete in {elapsed:.1f} seconds.")

DSM RASTER PROPERTIES
Bands: 3
Shape: (643, 942)
Reading elevation band.
Extracting elevation.
Elevation extracted for 28437 nodes.
Computing Sky View Factor.


Processing nodes: 100%|██████████| 28437/28437 [00:07<00:00, 3648.30it/s]

DSM extraction complete in 10.4 seconds.


## EXTRACT LANDCOVER FEATURES

In [ ]:
start_time = time.time()

# Initialize feature column.
df_nodes["landcover_class"] = np.nan

# Open landcover raster.
with rasterio.open(LANDCOVER) as src:
    print("LANDCOVER RASTER PROPERTIES")
    print(f"Shape: {src.shape}")
    print(f"CRS: {src.crs}")

    # Transform coordinates.
    xs, ys = rasterio.warp.transform("EPSG:4326", src.crs, lons, lats)

    # Convert to pixel indices.
    rows = []
    cols = []

    for x, y in zip(xs, ys):
        try:
            r, c = src.index(x, y)
            rows.append(r)
            cols.append(c)
        except:
            rows.append(-1)
            cols.append(-1)

    rows = np.array(rows)
    cols = np.array(cols)
    valid_mask = (rows >= 0) & (rows < src.height) & (cols >= 0) & (cols < src.width)

    print("Reading landcover band.")

    # Read landcover.
    lc_array = src.read(1)

    print("Extracting landcover classes.")

    # Extract landcover.
    lc_values = np.full(len(df_nodes), np.nan)
    lc_values[valid_mask] = lc_array[rows[valid_mask], cols[valid_mask]]

    if src.nodata is not None:
        lc_values[lc_values == src.nodata] = np.nan

    df_nodes["landcover_class"] = lc_values

elapsed = time.time() - start_time
print(f"Landcover extraction complete in {elapsed:.1f} seconds.")
print(f"Landcover: {df_nodes['landcover_class'].notna().sum()} / {len(df_nodes)} nodes")

LANDCOVER RASTER PROPERTIES
Shape: (11133, 11133)
CRS: EPSG:4326
Reading landcover band.
Extracting landcover classes.
Landcover extraction complete in 3.8 seconds.
Landcover: 28437 / 28437 nodes


## SAVE OUTPUT

In [ ]:
# Save network nodes with features.
df_nodes.to_csv(OUT_FILE, index = False)

print(f"Saved {len(df_nodes)} nodes to {OUT_FILE}.")
print(f"Columns: {list(df_nodes.columns)}")

Saved 28437 nodes to data/outputs/features/network_nodes_with_raster_features.csv.
Columns: ['osmid', 'y', 'x', 'street_count', 'highway', 'ref', 'railway', 'geometry', 'lon', 'lat', 'lst_kelvin', 'lst_celsius', 'ndvi', 'emissivity', 'palsar_hh_db', 'palsar_hv_db', 'palsar_hv_hh_ratio', 'palsar_glcm_contrast', 'palsar_glcm_homogeneity', 'palsar_glcm_energy', 'elevation_m', 'sky_view_factor', 'landcover_class']


## FEATURE SUMMARY

In [ ]:
print("NETWORK FEATURE EXTRACTION SUMMARY")
print(f"Total network nodes: {len(df_nodes)}")
print("")
print("Feature completeness:")

feature_cols = [
    "lst_celsius",
    "ndvi",
    "emissivity",
    "palsar_hh_db",
    "palsar_hv_db",
    "palsar_glcm_contrast",
    "elevation_m",
    "sky_view_factor",
    "landcover_class"
]

for col in feature_cols:
    if col in df_nodes.columns:
        valid = df_nodes[col].notna().sum()
        pct = valid / len(df_nodes) * 100
        print(f"{col:30s}: {valid:6d} ({pct:5.1f}%)")

NETWORK FEATURE EXTRACTION SUMMARY
Total network nodes: 28437

Feature completeness:
lst_celsius                   :  28332 ( 99.6%)
ndvi                          :  28437 (100.0%)
emissivity                    :  28437 (100.0%)
palsar_hh_db                  :  28437 (100.0%)
palsar_hv_db                  :  28437 (100.0%)
palsar_glcm_contrast          :  28437 (100.0%)
elevation_m                   :  28437 (100.0%)
sky_view_factor               :  28437 (100.0%)
landcover_class               :  28437 (100.0%)
